# AAT Keyword Embedding Pipeline

Embeds AAT terms from `KeeganCarey/aat-selectively-filtered` into a ChromaDB vector database with **multiple collections** for different embedding models.

Each collection lives in the same database so you can test different models side-by-side. The finished VDB is uploaded to Hugging Face for use by `mcam_server.ipynb`.

In [ ]:
%%capture
!pip install torch transformers chromadb datasets huggingface-hub hf_xet qwen-vl-utils sentencepiece protobuf open_clip_torch

In [ ]:
from huggingface_hub import login
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Logged in to Hugging Face')
else:
    print('HF_TOKEN not set — add it to Colab secrets or environment')

# Configuration

Edit `COLLECTIONS` to control which collections get built. Each entry maps a **collection name** to its embedding model. Comment out or add entries freely — only uncommented entries will be processed.

The `type` field selects which embedding loader to use (`"qwen"` or `"openclip"`).

In [ ]:
# ── Collection definitions ────────────────────────────────────────────
# Add, remove, or comment out entries to control which collections get built.
COLLECTIONS = [
    {
        "name": "aat_terms_qwen2b",
        "model_id": "Qwen/Qwen3-VL-Embedding-2B",
        "type": "qwen",
        "batch_size": 32,
    },
    # Uncomment to also build a SigLIP collection via OpenCLIP:
    # {
    #     "name": "aat_terms_siglip",
    #     "model_id": "ViT-SO400M-14-SigLIP-384",
    #     "pretrained": "webli",
    #     "type": "openclip",
    #     "batch_size": 128,
    # },
]

# ── Hugging Face repo for the vector database ──
HF_VDB_REPO = "KeeganCarey/mcam-vdb"

# ── Local build directory (Colab scratch space) ──
VDB_DIR = "/content/mcam-vdb"

# ── Source dataset ──
AAT_DATASET = "KeeganCarey/aat-selectively-filtered"

# ── Rebuild existing collections? ──
OVERWRITE = True

# Load AAT Dataset

Loads every term from the filtered AAT dataset and prepares:
- **text** — what gets embedded (term + scope note for richer semantics)
- **metadata** — every column stored alongside the embedding for filtering/display

In [ ]:
from datasets import load_dataset
import json

dataset = load_dataset(AAT_DATASET, split="train")
print(f"Loaded {len(dataset)} AAT terms")
print(f"Columns: {dataset.column_names}")

documents = []
for row in dataset:
    term = row["preferred_term"]
    scope = row["scope_note"] or ""
    text = f"{term}: {scope}" if scope else term

    metadata = {
        "subject_id":    row["subject_id"],
        "preferred_term": row["preferred_term"],
        "variant_terms":  json.dumps(row["variant_terms"]),  # list → JSON string
        "scope_note":     scope,
        "hierarchy":      row["hierarchy"],
        "facet":          row["facet"],
        "record_type":    row["record_type"],
        "parent_id":      row["parent_id"],
        "parent_term":    row["parent_term"],
        "sort_order":     row["sort_order"],
        "term_id":        row["term_id"],
        "root_id":        row["root_id"],
        "depth":          row["depth"],
        "child_count":    row["child_count"],
        "is_leaf":        row["is_leaf"],
    }

    documents.append({
        "id":       str(row["subject_id"]),
        "text":     text,
        "metadata": metadata,
    })

print(f"Prepared {len(documents)} documents")
print(f"Sample text: {documents[0]['text'][:120]}...")
print(f"Sample facet: {documents[0]['metadata']['facet']}  hierarchy: {documents[0]['metadata']['hierarchy']}")

# Embedding Model Loaders

Each model type has its own function that:
1. Loads the model
2. Embeds all texts in batches
3. Normalises embeddings to unit vectors (for cosine similarity)
4. Frees GPU memory when done

To add a new model, write a function with signature `(model_id, texts, batch_size) → list[list[float]]` and register it in `EMBED_FN`.

In [ ]:
import torch
import sys
import numpy as np
from tqdm import tqdm
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


def embed_qwen(model_id: str, texts: list[str], batch_size: int = 32, **kw) -> list[list[float]]:
    """Embed texts using a Qwen3-VL-Embedding model."""
    from huggingface_hub import hf_hub_download
    import importlib.util

    script = hf_hub_download(repo_id=model_id, filename="scripts/qwen3_vl_embedding.py")
    spec = importlib.util.spec_from_file_location("qwen3_vl_embedding", script)
    mod = importlib.util.module_from_spec(spec)
    sys.modules["qwen3_vl_embedding"] = mod
    spec.loader.exec_module(mod)

    try:
        import flash_attn
        attn = "flash_attention_2"
    except ImportError:
        attn = "sdpa"

    embedder = mod.Qwen3VLEmbedder(
        model_name_or_path=model_id,
        dtype=torch.float16 if device == "cuda" else torch.float32,
        attn_implementation=attn,
    )

    all_embeddings: list[list[float]] = []
    for i in tqdm(range(0, len(texts), batch_size), desc=f"Qwen ({model_id})"):
        batch = texts[i : i + batch_size]
        inputs = [{"text": t} for t in batch]
        features = embedder.process(inputs)
        features = torch.nn.functional.normalize(features, p=2, dim=1)
        all_embeddings.extend(features.cpu().float().numpy().tolist())

    del embedder
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    return all_embeddings


def embed_openclip(model_id: str, texts: list[str], batch_size: int = 128, **kw) -> list[list[float]]:
    """Embed texts using an OpenCLIP model (e.g. ViT-SO400M-14-SigLIP-384)."""
    import open_clip

    pretrained = kw.get("pretrained", "webli")
    model, _, preprocess = open_clip.create_model_and_transforms(model_id, pretrained=pretrained, device=device)
    tokenizer = open_clip.get_tokenizer(model_id)
    model.eval()

    all_embeddings: list[list[float]] = []
    for i in tqdm(range(0, len(texts), batch_size), desc=f"OpenCLIP ({model_id})"):
        batch = texts[i : i + batch_size]
        tokens = tokenizer(batch).to(device)
        with torch.no_grad(), torch.amp.autocast(device):
            features = model.encode_text(tokens)
            features = torch.nn.functional.normalize(features, p=2, dim=1)
        all_embeddings.extend(features.cpu().float().numpy().tolist())

    del model, tokenizer, preprocess
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    return all_embeddings


# ── Registry: add new model types here ──
EMBED_FN = {
    "qwen":     embed_qwen,
    "openclip": embed_openclip,
}

print(f"Registered embedding types: {list(EMBED_FN.keys())}")

# Build Vector Database

Iterates over every entry in `COLLECTIONS`, embeds the AAT terms with the specified model, and stores them in a ChromaDB collection with full metadata.

Each model is loaded, used, then freed before the next one starts — so even large models fit in a single GPU.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=VDB_DIR)

texts     = [doc["text"]     for doc in documents]
ids       = [doc["id"]       for doc in documents]
metadatas = [doc["metadata"] for doc in documents]

# Keys that are NOT passed to the embed function
_CFG_KEYS = {"name", "model_id", "type", "batch_size"}

for cfg in COLLECTIONS:
    name       = cfg["name"]
    model_id   = cfg["model_id"]
    model_type = cfg["type"]
    batch_size = cfg.get("batch_size", 32)
    # Extra keys (e.g. "pretrained") are forwarded to the embed function
    extra_kw   = {k: v for k, v in cfg.items() if k not in _CFG_KEYS}

    print(f"\n{'=' * 60}")
    print(f"Collection : {name}")
    print(f"Model      : {model_id}")
    print(f"Type       : {model_type}")
    if extra_kw:
        print(f"Extra      : {extra_kw}")
    print(f"{'=' * 60}")

    # Handle existing collection
    existing_names = [c.name for c in client.list_collections()]
    if name in existing_names:
        if OVERWRITE:
            client.delete_collection(name)
            print(f"Deleted existing collection '{name}'")
        else:
            print(f"Collection '{name}' exists — skipping (set OVERWRITE=True to rebuild)")
            continue

    # Embed
    embed_fn   = EMBED_FN[model_type]
    embeddings = embed_fn(model_id, texts, batch_size=batch_size, **extra_kw)
    dim        = len(embeddings[0])
    print(f"Generated {len(embeddings)} embeddings (dim={dim})")

    # Create collection
    collection = client.create_collection(
        name=name,
        metadata={"hnsw:space": "cosine"},
    )

    # Insert in chunks (ChromaDB batch limit)
    CHUNK = 5000
    for i in tqdm(range(0, len(ids), CHUNK), desc="Inserting"):
        collection.add(
            ids=ids[i : i + CHUNK],
            embeddings=embeddings[i : i + CHUNK],
            documents=texts[i : i + CHUNK],
            metadatas=metadatas[i : i + CHUNK],
        )

    print(f"Collection '{name}' — {collection.count()} documents stored")

print("\nAll collections built!")

# Verify Collections

In [ ]:
print("Collections in VDB:\n")
for col in client.list_collections():
    count = col.count()
    print(f"  {col.name}: {count} documents")

    sample = col.peek(1)
    if sample["ids"]:
        meta = sample["metadatas"][0]
        print(f"    Sample: {sample['documents'][0][:80]}...")
        print(f"    Facet={meta.get('facet')}  Hierarchy={meta.get('hierarchy')}")
    print()

# Upload to Hugging Face

Pushes the entire VDB directory (all collections) to the HF dataset repo. The server notebook downloads this with `snapshot_download`.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Create the repo if it doesn't exist yet
api.create_repo(repo_id=HF_VDB_REPO, repo_type="dataset", exist_ok=True)

# Upload the full VDB directory
api.upload_folder(
    repo_id=HF_VDB_REPO,
    repo_type="dataset",
    folder_path=VDB_DIR,
)

print(f"\nUploaded to https://huggingface.co/datasets/{HF_VDB_REPO}")
print("The server notebook can now download this VDB with snapshot_download.")